In [5]:
from networks import N3DED8
import torch
from torch.utils.data import DataLoader
from pytorch_datasets import SubjectIndependentTestDataset
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
## MODEL
RTRPPG = N3DED8()
RTRPPG.to(device)
checkpoint = torch.load('weights.pth.tar',map_location=torch.device('cpu'))
#print(checkpoint)

RTRPPG.load_state_dict(checkpoint['model_state_dict'])
current_path = os.getcwd()

In [8]:
## DATA MANAGER
from pytorch_datasets import SubjectIndependentTestDataset

path = os.path.abspath(os.path.join(current_path,'demo_subject/example'))

dataset = SubjectIndependentTestDataset(path) 
dataloader = DataLoader(dataset, batch_size=1, drop_last=False, shuffle=False)

In [9]:
GT = dataset.y_file # GT
time = dataset.t_file # time
window = dataset.window # Slinding window length
rPPG = []

with torch.no_grad():
    for idx, sample in enumerate(dataloader):
        out = RTRPPG(sample['x'])
        out = out - torch.mean(out,keepdim=True,dim=1) / torch.std(out,keepdim=True,dim=1)
        rPPG.append(out.to('cpu').detach().numpy())

rPPG = np.vstack(rPPG)

In [10]:
def normalize(data):    
    from sklearn.preprocessing import MinMaxScaler
    x = np.asarray(data)
    x = x.reshape(len(x), 1)
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler = scaler.fit(x)
    scaled_x = scaler.transform(x)
    return scaled_x

In [ ]:
## OVERLAP-ADD PROCESS                
y_hat = np.zeros(window+len(rPPG)-1)
for i in range(len(rPPG)):
    y_hat[i:i+window] = y_hat[i:i+window]+rPPG[i]
y_hat = np.squeeze(normalize(y_hat))

# PLOT
fig, ax = plt.subplots()
plt.plot(time,y_hat)
plt.ylabel("Amplitude")
plt.xlabel("Time [s]")
plt.legend(['rPPG'])     
fig.savefig('Output.png', format='png', dpi=1200)
print('[rtrppg demo]=>Output.png file saved in '+current_path)